# Generate Data

This notebook generates batches of simulated MRS spectra using the MRS phantom framework.
It performs the following steps:

1. Load a configuration file containing simulation parameters.
2. Split the main configuration into individual configurations for each simulation instance.
3. Run simulations for each configuration using the MRS phantom.
4. Save the results (e.g., spectra, ground truth data) for further processing or validation.

> **Note:** Use this notebook when you want to generate a new dataset of simulated MRS spectra in a reproducible and configurable way.


## Imports

In [1]:
# === Library Imports ===
import json                                         # For loading and parsing configuration files
import sys                                          # For system-specific parameters and functions
from tqdm import tqdm                               # Progress bar for long-running loops
import os                                           # For file and directory operations
from PyQt5.QtWidgets import QApplication            # GUI backend required for voxel selector

# === Set up the project root directory ===
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(PROJECT_ROOT)

# === Local Simulation Core ===
from simulation.mrs_phantom import DigitalPhantom   # Core class for generating phantom-based MRS data
from simulation.run_matlab import MatlabRunner      # Interface to run MATLAB components (if needed)

# === Local GUI Tools ===
from gui.main_voxel_selector import PhantomWindow   # GUI for interactive voxel selection
from gui.defaults import *                          # Default GUI settings (window size, paths, etc.)

# === Local Utility Functions ===
from utils.auxillary import (
    split_config_into_simulations,                  # Break main config into per-simulation configs
    group_simulation_configs_by_settings,           # Group similar configs for efficient processing
    unflatten_dict                                  # Reconstruct nested dictionaries from flattened ones
)
from utils.nii_processing import save_nifti_mrs     # Save MRS data in NIfTI-MRS format

## Settings

In [2]:
# === Load initial configuration ===

# Path to the main configuration file
config_path = './config.json'

# Flag to enable or disable manual voxel selection via GUI
select_voxel = True

# Load the configuration from the JSON file
with open(config_path, 'r') as file:
    config = json.load(file)


## Optional: Launch Voxel Selection GUI

In [3]:
# === Optional: Launch voxel selection GUI ===
# Initialize a DigitalPhantom object using the skeleton provided in the config
# This is only needed when GUI-based voxel selection is enabled

if select_voxel:
    phantom = DigitalPhantom(
        skeleton=config['skeleton'],               # Tissue label + anatomical information
        path2metabs=DEFAULT_METAB_DF_PATH,         # Default metabolite database
        path2basis=DEFAULT_path2basis,             # Default basis spectra path
        simulation_params=DEFAULT_SIM_PARAMS,      # Default simulation parameters
    )

    # Initialize Qt application and show the Phantom GUI window
    app = QApplication([])                         # Create an empty QApplication (required for GUI)
    window = PhantomWindow(phantom, config, output_dir='outputs')
    window.show()                                  # Launch the GUI
    app.exec_()                                    # Enter the event loop until the GUI is closed

    # Update the config path with the file generated by the GUI
    config_path = window.config_path


[INFO] 📥 Initializing Digital Phantom...
Metabolites to ignore from basisset: [np.str_('Ala'), np.str_('Tyros'), np.str_('Phenyl'), np.str_('Ref0ppm'), np.str_('Thr'), np.str_('EA'), np.str_('Ser'), np.str_('Cit'), np.str_('EtOH')]
[INFO] 🧬 Generating lipid mask from explicit fat label...
Added voxel: (74, 104, 161, 176, 68, 93) with size (30, 15, 25), size in mm: (30.0, 15.0, 25.0)
Saving selected voxel definitions to new config file...
Saved updated config file to: outputs/simulation_20260213_se0w/config.json


# Run Simulations from Config Files
This block runs the batch simulation process. It:

1. Splits the main config into per-voxel simulation configs.
2. Groups those configs by simulation settings (to reuse basis sets).
3. Checks or generates required basis sets.
4. Initializes the DigitalPhantom.
5. Simulates MRS data for each voxel and saves spectra as NIfTI-MRS files.

In [4]:
# === Step 1: Split main config into individual simulation configs ===
split_config_into_simulations(config_path)
print("Simulation configurations generated successfully.")

# === Step 2: Group configs by shared simulation/basis settings ===
groups = group_simulation_configs_by_settings(config_path)

idx = 0  # Spectrum index counter for naming
# === Step 3: Loop over each group of settings ===
for settings_key, config_files in tqdm(groups.items(), desc="Processing settings groups", unit="group"):
    
    settings = unflatten_dict(settings_key)
    print(f"\nInitializing simulation with settings: {settings_key}")

    # === Step 4: Determine basis set path ===
    if 'path2basis' in settings:
        # Use provided basis set path
        basis_path = settings['path2basis']
        
        if os.path.exists(basis_path):
            print(f"Using provided basis set: {basis_path}")
        else:
            print(f"Provided basis set path does not exist: {basis_path}")
            print("Falling back to default basis set generation...")

            # Compose expected basis filename
            expected_file_name = f"LCModel_{settings['basis_set_settings']['vendor']}_UnEdited_" \
                                 f"{settings['basis_set_settings']['localization']}_TE" \
                                 f"{settings['basis_set_settings']['TE']}.BASIS"
            basis_path = os.path.join(settings['basis_set_dir'], expected_file_name)

            if os.path.exists(basis_path):
                print(f"Found fallback basis set: {basis_path}")
            else:
                print(f"Generating basis set: {expected_file_name}")
                matlab_runner = MatlabRunner()
                matlab_runner.generate_basis_set(settings['basis_set_settings'])
                print(f"Basis set generated: {basis_path}")

    else:
        # No basis path provided, generate expected filename and check
        expected_file_name = f"LCModel_{settings['basis_set_settings']['vendor']}_UnEdited_" \
                             f"{settings['basis_set_settings']['localization']}_TE" \
                             f"{settings['basis_set_settings']['TE']}.BASIS"
        basis_path = os.path.join(settings['basis_set_dir'], expected_file_name)

        if os.path.exists(basis_path):
            print(f"Using existing basis set: {basis_path}")
        else:
            print(f"Basis set not found: {basis_path}")
            print("Generating new basis set...")
            matlab_runner = MatlabRunner()
            matlab_runner.generate_basis_set(settings['basis_set_settings'])
            print(f"Basis set generated: {basis_path}")

    # === Step 5: Combine simulation and basis parameters ===
    sim_parms = settings['simulation_params']
    basis_params = settings['basis_set_settings']
    all_params = {**sim_parms, **basis_params}

    # === Step 6: Initialize phantom with current settings ===
    phantom = DigitalPhantom(
        skeleton=settings['skeleton'],
        path2metabs=settings['path2metabs'],
        path2basis=basis_path,
        simulation_params=all_params,
        gui=None,  # GUI not used during batch simulations
        metabs=settings['metabs'],
    )

    # === Step 7: Set save directory ===
    save_dir = os.path.join(os.path.dirname(config_path), 'spectra')
    os.makedirs(save_dir, exist_ok=True)

    # === Step 8: Loop over each voxel config in the group ===
    for file in tqdm(config_files, desc=f"Simulating voxels for group {idx}", unit="voxel", leave=False):
        with open(file, 'r') as f:
            config = json.load(f)

        voi_coords = config['voxel_definitions']['coords']

        # Run the simulation
        spec, components, t, ppm = phantom.simulate_data(voi_coords)
        # Option A — per-tissue sampled concentrations
        sampled = phantom.signal_model.sampled_concs_per_tissue

        # Option B — final voxel-weighted concentrations
        voxel_final = phantom.signal_model.final_voxel_concentrations

        fractions = phantom.signal_model.voxel_tissue_fractions

        
        print("Sampled tissue concs:", sampled)
        print("Final voxel concs:", voxel_final)
        print("Tissue fractions:", fractions)


        # === Step 9: Save result as NIfTI-MRS file ===
        save_nifti_mrs(
            save_dir,
            total_selection=["Metabs", "MMs", "Lipids", "Water", "Noise"],
            individual_selection=["Metabs", "MMs", "Lipids", "Water", "Noise"],
            components_data=components,
            affine=phantom.affine,
            dwelltime=phantom.basis_set.dwelltime,
            spec_freq=phantom.basis_set.cf,
            sim_params=config,
            base_name=f"simulated_spectrum_{idx}",
        )


        conc_save_path = os.path.join(save_dir, f"concentrations_{idx}.json")

        with open(conc_save_path, "w") as f:
            json.dump({
                "sampled_per_tissue": sampled,
                "voxel_weighted": voxel_final, 
                "tissue_fractions": fractions
            }, f, indent=2)

            
        idx += 1
        # Clean up temporary config file
        os.remove(file)


print("✅ All simulations completed and spectra saved!")


Saved config outputs/simulation_20260213_se0w/simulation_config_0.json
Saved config outputs/simulation_20260213_se0w/simulation_config_1.json
Saved config outputs/simulation_20260213_se0w/simulation_config_2.json
Saved config outputs/simulation_20260213_se0w/simulation_config_3.json
Generated 4 simulation configuration files.
Simulation configurations generated successfully.


Processing settings groups:   0%|                                                                                                                              | 0/4 [00:00<?, ?group/s]


Initializing simulation with settings: (('basis_set_dir', '/home/arijitbhattacharya/osprey-develop/fit/basissets/'), ('basis_set_settings.TE', '30'), ('basis_set_settings.localization', 'PRESS'), ('basis_set_settings.vendor', 'Universal_Seimens'), ('metabs', '[]'), ('path2basis', '/home/arijitbhattacharya/osprey-develop/fit/basissets/3T/siemens/unedited/press/30/3T_press_Siemens_30ms_noMM_A.BASIS'), ('path2metabs', '/home/arijitbhattacharya/MRS-Digital-Phantom/data/metabolites/metab_df.csv'), ('simulation_params.TR', '2000.0'), ('simulation_params.bandwidth', '2000.0'), ('simulation_params.lipid_amp_factor', '10.0'), ('simulation_params.lipid_lw_max', '50.0'), ('simulation_params.lipid_lw_min', '30.0'), ('simulation_params.lipid_phase_max', '180.0'), ('simulation_params.lipid_phase_min', '-180.0'), ('simulation_params.lipid_sigma', '5.0'), ('simulation_params.mm_json_file', './data/macromolecules/mm_params.json'), ('simulation_params.mm_level', '10.0'), ('simulation_params.noise_level


Simulating voxels for group 0:   0%|                                                                                                                           | 0/1 [00:00<?, ?voxel/s]

⚠️ No B0 map provided, simulating shim imperfections...
Simulating metabolites...
Removing metabolites from dataframe that are not in the basis set...
1 metabolites removed: {'NAA_Asp'}
Simulating macromolecules...
Simulating lipids...
Loading lipid basis set from cache: ./simulation/lipids/lipids_cache/756e13a28f53679cab12c1d007f76aa5.mat
Simulating water...
Simulating noise...
✅ Simulation complete!

=== Sampled concentrations per tissue ===
{'Background': {'Asc': 0.0, 'Asp': 0.0, 'Cr': 0.0, 'GABA': 0.0, 'Glc': 0.0, 'Gln': 0.0, 'Glu': 0.0, 'Gly': 0.0, 'GPC': 0.0, 'GSH': 0.0, 'Lac': 0.0, 'mI': 0.0, 'NAA': 0.0, 'NAAG': 0.0, 'PCh': 0.0, 'PCr': 0.0, 'PE': 0.0, 'sI': 0.0, 'Tau': 0.0}, 'WM': {'Asc': 0.0, 'Asp': 1.6281396520911828, 'Cr': 5.192429456171656, 'GABA': 0.9895695812425694, 'Glc': 0.0, 'Gln': 2.7378028637428904, 'Glu': 8.555199248792398, 'Gly': 0.0, 'GPC': 1.1986132565006669, 'GSH': 0.9747218389829246, 'Lac': 0.0, 'mI': 5.696848347691394, 'NAA': 8.344740748259245, 'NAAG': 2.141257

...

Simulating noise...
✅ Simulation complete!

=== Sampled concentrations per tissue ===
{'Background': {'Asc': 0.0, 'Asp': 0.0, 'Cr': 0.0, 'GABA': 0.0, 'Glc': 0.0, 'Gln': 0.0, 'Glu': 0.0, 'Gly': 0.0, 'GPC': 0.0, 'GSH': 0.0, 'Lac': 0.0, 'mI': 0.0, 'NAA': 0.0, 'NAAG': 0.0, 'PCh': 0.0, 'PCr': 0.0, 'PE': 0.0, 'sI': 0.0, 'Tau': 0.0}, 'WM': {'Asc': 0.0, 'Asp': 1.9969482371071794, 'Cr': 5.221926031246072, 'GABA': 1.1631098068841452, 'Glc': 0.0, 'Gln': 2.485089046907411, 'Glu': 9.12627638221082, 'Gly': 0.0, 'GPC': 1.0458417606666184, 'GSH': 1.0325388609422672, 'Lac': 0.0, 'mI': 5.042392818752239, 'NAA': 7.0291597023219605, 'NAAG': 2.3195923152031304, 'PCh': 0.5601353827370319, 'PCr': 2.347136722808771, 'PE': 0.0, 'sI': 0.5549734124544793, 'Tau': 1.1829833152471845}, 'GM': {'Asc': 1.3373988735966207, 'Asp': 3.056557650763659, 'Cr': 5.293688287567703, 'GABA': 1.7974799640967063, 'Glc': 0.0, 'Gln': 2.3651986682729174, 'Glu': 7.809353714860416, 'Gly': 4.10296539788835, 'GPC': 0.7180562144770002, 'GS


Simulating voxels for group 3:   0%|                                                                                                                           | 0/1 [00:00<?, ?voxel/s]

⚠️ No B0 map provided, simulating shim imperfections...
Simulating metabolites...
Removing metabolites from dataframe that are not in the basis set...
1 metabolites removed: {'NAA_Asp'}
Simulating macromolecules...
Simulating lipids...
Loading lipid basis set from cache: ./simulation/lipids/lipids_cache/756e13a28f53679cab12c1d007f76aa5.mat
Simulating water...



Processing settings groups: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:36<00:00,  9.11s/group]

Simulating noise...
✅ Simulation complete!

=== Sampled concentrations per tissue ===
{'Background': {'Asc': 0.0, 'Asp': 0.0, 'Cr': 0.0, 'GABA': 0.0, 'Glc': 0.0, 'Gln': 0.0, 'Glu': 0.0, 'Gly': 0.0, 'GPC': 0.0, 'GSH': 0.0, 'Lac': 0.0, 'mI': 0.0, 'NAA': 0.0, 'NAAG': 0.0, 'PCh': 0.0, 'PCr': 0.0, 'PE': 0.0, 'sI': 0.0, 'Tau': 0.0}, 'WM': {'Asc': 0.0, 'Asp': 1.702301872436586, 'Cr': 5.564612221969968, 'GABA': 1.2182467475402061, 'Glc': 0.0, 'Gln': 2.8708218203740548, 'Glu': 8.754146941748612, 'Gly': 0.0, 'GPC': 0.9294560590158416, 'GSH': 1.2471522411155116, 'Lac': 0.0, 'mI': 3.9829767121412187, 'NAA': 8.004739624077114, 'NAAG': 2.577149535203467, 'PCh': 0.7252793432278362, 'PCr': 1.902093195462541, 'PE': 0.0, 'sI': 0.5080013416983934, 'Tau': 2.572259384398224}, 'GM': {'Asc': 1.563241156750764, 'Asp': 2.162251975338995, 'Cr': 5.537914293546718, 'GABA': 1.8828447612624433, 'Glc': 0.0, 'Gln': 2.5825092088066333, 'Glu': 8.028438138280833, 'Gly': 6.773729458708802, 'GPC': 0.6771389093179596, 'GSH